# Use case: temporal variability in 14 years of BRFSS with `dashi`

**Scenario.** The Behavioral Risk Factor Surveillance System (BRFSS) is the US CDC's yearly telephone health
survey, with around 400,000 adults interviewed every year. Suppose we want to train a model that predicts
whether a respondent has ever been told they have diabetes, and keep using it as new survey waves arrive.
Before doing so we need to know *whether the data are stable over time*: do the population, the way the
questions are recorded, and the relationship between risk factors and diabetes stay the same?

This notebook answers those questions with the temporal tools of `dashi`:

| Question | `dashi` tool |
|---|---|
| How does each variable's distribution change month to month? | Data Temporal Maps (DTMs) and their heatmaps |
| Are the changes gradual trends, seasonal, or abrupt breaks? | Information Geometric Temporal (IGT) projections |
| Does the combination of all covariates drift? (**covariate shift**) | Multivariate DTMs |
| Does the diabetes prevalence drift? (**prior shift**) | DTM of the label |
| Do the changes remain within each diagnosis? (**conditional shift**) | Label-conditional DTMs |
| Does a model trained in one year keep working in later years? | Multi-batch models and performance matrices |

**Requirements.** `dashi >= 0.4.3` (`pip install -U dashi`). Run the notebook from the `examples` directory.

The full notebook takes a few minutes, most of it in the supervised section.

**Data.** `brfss_diabetes.csv` is a random sample of 1,000 respondents per interview month, January 2011 to
December 2024 (168,000 rows), drawn from the public BRFSS annual files (https://www.cdc.gov/brfss/). It keeps 22
questions of the fixed core that were asked every year, harmonised across the yearly renames, with their original
codes (don't know / refused stored as missing). The label `diabetes` is 1 for "ever told you have diabetes"
and 0 otherwise; pre-diabetes and diabetes only during pregnancy count as 0.

In [ ]:
from pathlib import Path

import dashi as ds
import numpy as np
import pandas as pd
import plotly.io as pio

pd.set_option("display.max_columns", 30)
# Keep a light background even when the editor uses a dark theme.
pio.templates.default = "plotly_white"

## 1. Load and format the data

In [ ]:
DATA_PATH = Path("brfss_diabetes.csv")

raw = pd.read_csv(DATA_PATH)
print(f"{len(raw):,} respondents, {raw['date'].min()} to {raw['date'].max()}")
raw.head()

`format_data` parses the interview month and casts every column to the type `dashi` expects. All survey
answers are coded, so they are declared as categorical; BMI and the number of physically / mentally unhealthy
days in the last month are numerical. Missing answers of categorical variables become their own category
(`'nan'`), so changes in missingness are analysed like any other change.

In [ ]:
DATE = "date"
LABEL = "diabetes"
numerical_covariates = ["bmi", "physhlth", "menthlth"]
categorical_covariates = [
    "age_group", "sex", "race", "education", "income", "marital", "genhlth", "checkup", "medcost",
    "exercise", "smoker", "binge", "stroke", "heart", "asthma", "copd", "depression", "kidney", "arthritis",
]
covariates = categorical_covariates + numerical_covariates

data = ds.format_data(
    raw,
    date_column_name=DATE,
    date_format="%Y-%m-%d",
    numerical_column_names=numerical_covariates,
    categorical_column_names=[LABEL] + categorical_covariates,
)
data.dtypes.rename("dtype").to_frame().T

## 2. A first look by year

Yearly summaries already hint at some changes, but they cannot tell a gradual trend from an abrupt break, and
they do not scale to inspecting every category of every variable.

In [ ]:
year = raw[DATE].str[:4]
overview = pd.DataFrame({
    "diabetes_prevalence": raw[LABEL].groupby(year).mean(),
    "missing_income": raw["income"].isna().groupby(year).mean(),
    "missing_exercise": raw["exercise"].isna().groupby(year).mean(),
    "missing_bmi": raw["bmi"].isna().groupby(year).mean(),
}).rename_axis("year")
overview.T.round(3)

## 3. Unsupervised characterisation: Data Temporal Maps

A Data Temporal Map estimates the probability distribution of every variable in every period (here, month) over
a common support. The `'nan'` category of the categorical variables is included, so changes in missingness are
visible too.

In [ ]:
temporal_maps = ds.estimate_univariate_data_temporal_map(
    data=data,
    date_column_name=DATE,
    period="month",
    numeric_smoothing=True,
)
print(f"{len(temporal_maps)} maps, {len(temporal_maps[LABEL].dates)} months each")

The heatmap of a DTM shows time on the horizontal axis, the variable's values on the vertical axis and the
probability as colour. The income map shows an abrupt change in January 2021:

In [ ]:
ds.plot_univariate_data_temporal_map(
    temporal_maps["income"], title="Income group by interview month",
).show()

From 2021 the survey replaced its income variable: the old top group (≥ 50k, code 5) was split into
50-100k (5), 100-200k (6) and ≥ 200k (7). The data are not "wrong", but code 5 no longer means the same
thing, so a model trained before 2021 would misread every later respondent. This is a **coding change** of the
kind that `dashi` makes visible at a glance.

The exercise map shows a different pattern: a yearly alternation of missing answers.

In [ ]:
ds.plot_univariate_data_temporal_map(
    temporal_maps["exercise"], title="Any exercise in the last month (1 = yes, 2 = no)",
).show()

In the odd years from 2011 to 2019, 5-8% of the exercise answers are missing, against less than 1% in the other
years, and the missing answers come out of the "yes" share. This is a change in how the answer was collected
or recorded, not in the population. Other variables drift slowly instead, for instance BMI and the diabetes label
itself (**prior shift**: the prevalence rises from about 12% to 14%):

In [ ]:
ds.plot_univariate_data_temporal_map(
    temporal_maps["bmi"], title="Body mass index by interview month",
).show()

ds.plot_univariate_data_temporal_map(
    temporal_maps[LABEL], mode="series", title="Diabetes (1) vs no diabetes (0) by interview month",
).show()

## 4. Information Geometric Temporal (IGT) projections

An IGT projection places every month as a point so that the distance between two points approximates the
Jensen-Shannon distance between their distributions. Similar months cluster together; a trend draws a path; an
abrupt change shows up as a gap between groups of points. Points are coloured by season.

In [ ]:
income_igt = ds.estimate_igt_projection(temporal_maps["income"], dimensions=2)
ds.plot_IGT_projection(income_igt, dimensions=2, trajectory=True).show()

The months before and after 2021 form two separate groups: the coding change is by far the largest movement in
the income variable. The exercise projection splits into "odd year" and "even year" groups instead:

In [ ]:
exercise_igt = ds.estimate_igt_projection(temporal_maps["exercise"], dimensions=2)
ds.plot_IGT_projection(exercise_igt, dimensions=2).show()

### 4.1 Scanning every variable

With 23 variables it is convenient to rank them before plotting. The `projection` attribute of an IGT
projection holds the coordinates of every month, so we can summarise each variable by:

- `drift`: the distance between the centre of the first year (2011) and the centre of the last year (2024);
- `largest_jump`: the largest month-to-month step relative to the typical (median) step, and when it happens.

These are simple summaries of the projection written for this notebook, not `dashi` metrics.

In [ ]:
def summarize_igt(igt_projection):
    points = np.asarray(igt_projection.projection)
    dates = pd.to_datetime(igt_projection.data_temporal_map.dates)
    steps = np.linalg.norm(np.diff(points, axis=0), axis=1)
    largest = int(np.argmax(steps))
    return {
        "drift": float(np.linalg.norm(points[:12].mean(axis=0) - points[-12:].mean(axis=0))),
        "largest_jump": float(steps[largest] / np.median(steps)),
        "largest_jump_month": dates[largest + 1].strftime("%Y-%m"),
    }


igt_projections = {
    variable: ds.estimate_igt_projection(temporal_maps[variable], dimensions=2)
    for variable in [LABEL] + covariates
}
igt_summary = pd.DataFrame(
    {variable: summarize_igt(igt) for variable, igt in igt_projections.items()}
).T.sort_values("drift", ascending=False)
igt_summary.round(3).head(10)

Income stands out on both criteria. Exercise has jumps at the start of each year, and smoking status, age, check-up
frequency and marital status drift slowly, which reflects a changing survey population (for example, an older
sample with fewer smokers and more yearly check-ups). Very rare conditions such as stroke barely move.

## 5. Fixing the coding change and checking again

Merging the three new income groups back into the old "≥ $50k" group makes the variable mean the same thing
across the whole period. Re-estimating the DTM checks that the fix worked:

In [ ]:
harmonised = data.copy()
# Categories are stored as strings by format_data; missing answers stay as 'nan'.
harmonised["income"] = (
    harmonised["income"].astype(str).replace({"6.0": "5.0", "7.0": "5.0"}).astype("category")
)

income_map_harmonised = ds.estimate_univariate_data_temporal_map(
    data=harmonised[[DATE, "income"]], date_column_name=DATE, period="month",
)
ds.plot_univariate_data_temporal_map(
    income_map_harmonised, title="Income group by interview month, harmonised",
).show()

income_igt_harmonised = ds.estimate_igt_projection(income_map_harmonised, dimensions=2)
pd.DataFrame({
    "original": summarize_igt(income_igt),
    "harmonised": summarize_igt(income_igt_harmonised),
}).T

The break is gone: the largest monthly jump drops from about 11 to about 3 times the typical step, and what is left is a
gradual shift towards higher incomes. The rest of the notebook uses the harmonised data.

## 6. Joint (multivariate) covariate variability

The multivariate DTM embeds all covariates with Factor Analysis of Mixed Data (FAMD, suitable for numerical plus
categorical variables) and estimates the density of every month in the reduced space. Dimensionality reduction
needs numerical values, so respondents with a missing BMI or unhealthy-days count are left out; missing answers
of categorical questions stay as their own category.

In [ ]:
joint_data = harmonised[[DATE] + covariates].dropna()
print(f"Respondents with complete numerical covariates: {len(joint_data):,} of {len(harmonised):,}")

joint_map = ds.estimate_multivariate_data_temporal_map(
    data=joint_data,
    date_column_name=DATE,
    period="month",
    dim_reduction="FAMD",
    dimensions=2,
    kde_resolution=20,
    verbose=True
)
ds.plot_multivariate_data_temporal_map(joint_map).show()

In [ ]:
joint_igt = ds.estimate_igt_projection(joint_map, dimensions=3)
ds.plot_IGT_projection(joint_igt, dimensions=3, trajectory=False).show()

Once the income break is removed, the joint covariate distribution follows a continuous path from 2011 to 2024
with no single dominant break (the largest monthly step is about 3 times the typical one), and a somewhat faster
move around 2020-2021. The population covered by the survey changes **gradually**, so models should expect slow
covariate drift rather than one abrupt shift.

## 7. Conditional variability: does the drift remain within each diagnosis?

The prevalence of diabetes also rises over time, so part of the covariate drift might be a consequence of that
prior shift. The label-conditional DTM estimates the joint covariate distribution separately for respondents
with and without diabetes; `estimate_igt_projection` combines both.

In [ ]:
conditional_joint_data = harmonised[[DATE, LABEL] + covariates].dropna()

conditional_joint_map = ds.estimate_conditional_data_temporal_map(
    data=conditional_joint_data,
    date_column_name=DATE,
    label_column_name=LABEL,
    period="month",
    dim_reduction="FAMD",
    dimensions=3,
    kde_resolution=20,
)
for figure in ds.plot_conditional_data_temporal_map(conditional_joint_map):
    figure.show()

conditional_igt = ds.estimate_igt_projection(conditional_joint_map, dimensions=3)
ds.plot_IGT_projection(conditional_igt, dimensions=3, trajectory=False).show()

The temporal path is still there within each diagnosis, so the drift is not only explained by the rising
prevalence: respondents with the same diagnosis also look different in 2024 than in 2011.

## 8. Supervised characterisation: does a model keep working over time?

`estimate_multibatch_models` splits the data into yearly batches, trains a model on each year and tests it on a
held-out 20% of **every** year. Two strategies are compared:

- `from_scratch`: every model is trained on a single year;
- `cumulative`: every model is trained on all years up to its training year (2011 to 2011, 2011 to 2012, ...).

Each model is a Random Forest that predicts diabetes from all covariates of the harmonised data, using the
respondents with complete numerical covariates (as in section 6).

**Holding out every month, not only the last ones.** By default, `estimate_multibatch_models` sorts the rows
of each batch by date and holds out the *last* 20%. With monthly dates, every test set would then contain only
the October to December interviews. `shuffle=True` holds out a random 20% of each year instead (fixed seed,
stratified by label), so the test sets cover all months.

In [ ]:
supervised_data = harmonised[[DATE, LABEL] + covariates].dropna()
print(f"Respondents used: {len(supervised_data):,}")

model_settings = dict(
    data=supervised_data,
    inputs_numerical_column_names=numerical_covariates,
    inputs_categorical_column_names=categorical_covariates,
    output_classification_column_name=LABEL,
    date_column_name=DATE,
    period="year",
    model_type="random_forest",
    shuffle=True,
)
performance_single_year = ds.estimate_multibatch_models(learning_strategy="from_scratch", **model_settings)
performance_cumulative = ds.estimate_multibatch_models(learning_strategy="cumulative", **model_settings)

In [ ]:
ds.plot_multibatch_performance(metrics=performance_single_year, metric_name="AUC_MACRO").show()
ds.plot_multibatch_performance(metrics=performance_cumulative, metric_name="AUC_MACRO").show()

To separate the effect of the training year from that of the test year, we average the AUC matrix by row
(how good a training year is, over all test years) and by column (how hard a test year is, for all models).

In [ ]:
auc_single_year = ds.arrange_performance_metrics(metrics=performance_single_year, metric_name="AUC_MACRO")
auc_single_year.index = [label.removeprefix("From ").split(" to ")[0] for label in auc_single_year.index]
auc_cumulative = ds.arrange_performance_metrics(metrics=performance_cumulative, metric_name="AUC_MACRO")

pd.DataFrame({
    "mean_AUC_by_training_year": auc_single_year.mean(axis=1).values,
    "mean_AUC_by_test_year": auc_single_year.mean(axis=0).values,
    "cumulative_model_up_to_year": auc_cumulative.mean(axis=1).values,
}, index=auc_single_year.columns.rename("year")).round(3).T

**Reading the matrices.** Rows are training years (or the last year of the cumulative window) and columns are
test years.

- All models reach a macro AUC of about 0.78-0.84, and **the training year barely matters**: the row averages
  only range from 0.802 to 0.813, so a model trained in 2011 is still about as good in 2024 as one trained in 2024.
  The most recent training years are slightly lower (0.802-0.805 from 2020 on).
- **The test year matters more, but mostly by chance.** The column averages range from 0.788 (2021) to 0.827
  (2017). Every model is scored on the same held-out respondents of a test year (about 2,100 people), so a
  held-out sample that happens to be hard looks hard for every model. Splitting with a different seed changes
  which years come out hardest, so differences of 0.01-0.03 between test years should not be over-read.
- **Cumulative training** helps slightly: about +0.01 AUC for the later years, where single-year models are
  weakest.

So, for this task, the gradual drift found by the unsupervised analysis does not make models go stale. AUC
does not measure calibration, though: with a rising prevalence, predicted probabilities and decision thresholds
may still need updating. `dashi` reports other metrics (for example `LOGLOSS` or `RECALL_1`) that can be
plotted the same way.

The small decline of the recent training years raises a question that a single hold-out split cannot answer:
are the recent data intrinsically harder to separate? The next section looks at that with all the data.

### 8.1 Do the predictors separate the two classes less well over time?

A model can only tell respondents with and without diabetes apart if their covariates differ. The
label-conditional univariate DTM estimates, for every month, the distribution of a variable separately for each
class, so we can watch both distributions move. We look at three of the strongest predictors: BMI, age group and
self-rated general health (1 = excellent, 5 = poor).

In [ ]:
separation_variables = ["bmi", "age_group", "genhlth"]

class_maps = ds.estimate_conditional_univariate_data_temporal_map(
    data=harmonised[[DATE, LABEL] + separation_variables],
    date_column_name=DATE,
    label_column_name=LABEL,
    period="month",
)
print("Labels:", list(class_maps))

The conditional heatmaps show the two classes one above the other (0 = no diabetes, 1 = diabetes):

In [ ]:
ds.plot_conditional_univariate_data_temporal_map(
    class_maps, variable_name="bmi", title="Body mass index by diabetes status",
).show()
ds.plot_conditional_univariate_data_temporal_map(
    class_maps, variable_name="genhlth", sorting_method="joint_frequency",
    title="Self-rated general health by diabetes status",
).show()
ds.plot_conditional_univariate_data_temporal_map(
    class_maps, variable_name="age_group", sorting_method="joint_frequency",
    title="Age group by diabetes status",
).show()

To measure how far apart the two classes are, we compute the Jensen-Shannon distance between the two
conditional distributions of every month (0 = identical, 1 = no overlap), and average it by year. This is a
simple summary of the maps' `probability_map` attribute written for this notebook, not a `dashi` metric.

In [ ]:
from scipy.spatial.distance import jensenshannon


def class_separation(maps_by_label, variable):
    no_diabetes = maps_by_label["0"][variable]
    diabetes = maps_by_label["1"][variable]
    distances = [
        jensenshannon(p, q)
        for p, q in zip(np.asarray(no_diabetes.probability_map), np.asarray(diabetes.probability_map))
    ]
    return pd.Series(distances, index=pd.to_datetime(no_diabetes.dates).year)


separation = pd.DataFrame({
    variable: class_separation(class_maps, variable).groupby(level=0).mean()
    for variable in separation_variables
}).rename_axis("year")
separation.T.round(3)

The IGT projection of each class shows how that class's distribution moves over the months. Plotting both classes
for the same variable shows whether they move together or drift apart:

In [ ]:
for variable in ["bmi", "genhlth"]:
    for label, name in [("0", "no diabetes"), ("1", "diabetes")]:
        igt = ds.estimate_igt_projection(class_maps[label][variable], dimensions=2)
        figure = ds.plot_IGT_projection(igt, dimensions=2, trajectory=False)
        figure.update_layout(title=f"IGT projection of {variable}, {name}")
        figure.show()
        print(f"{variable}, {name}: drift 2011 -> 2024 = {summarize_igt(igt)['drift']:.3f}")

**What the maps show.**

- **BMI separates the classes less and less.** The distance between the two BMI distributions falls from 0.26
  in 2011 to 0.20 in 2023-2024, the lowest values of the period. The IGT projections show why: the BMI of
  respondents *without* diabetes drifts steadily upwards (IGT drift 0.065), while the BMI of respondents *with*
  diabetes barely moves (0.016). The non-diabetic population is catching up, so BMI says less about who has
  diabetes.
- **General health follows the same trend**, from 0.31 to 0.26-0.28 in 2022-2024. Here both classes drift by a
  similar amount, but the two distributions get closer.
- **Age group shows no trend** (0.27-0.31 throughout).

The weakening is **gradual**, not specific to one year: it is a slow change in the relationship between the
risk factors and the label (a **conditional shift**). It lowers what any model can achieve with these
covariates, which fits the slightly lower AUC of the most recent training years. It is small next to the
chance variation of a single test year, which is why it shows clearly in the maps, computed from all the data,
but not in the performance matrices. The data cannot say why BMI and diabetes are becoming less related; one
hypothesis worth checking with other sources is the wider use of weight-lowering diabetes treatments in recent
years.

## 9. Take-home messages

1. **Coding changes are the largest shifts.** The 2021 income recoding and the alternating missingness of the
   exercise question are abrupt and survey-made. DTM heatmaps and IGT projections expose them immediately, and
   the income fix can be checked by re-running the same analysis.
2. **The population drifts gradually.** Once the recoding is fixed, the joint covariate distribution moves
   slowly and continuously, and the drift remains within each diagnosis.
3. **Prior shift is moderate.** Diabetes prevalence rises from about 12% to 14% over the period.
4. **Shift does not always mean degradation.** For this task, models trained on older years keep a similar
   discrimination in later years, and the differences between test years are mostly due to their held-out
   samples.
5. **The predictors slowly lose information.** Label-conditional maps show that BMI and general health separate
   respondents with and without diabetes less well over time, mainly because the BMI of respondents without
   diabetes rises. This conditional shift lowers what any model can achieve in recent years.
   Monitoring both the data (unsupervised) and the performance (supervised) is what allows these conclusions.

The data are an unweighted sample of a complex survey, so the distributions describe the sample, not the US
population. The supervised results come from one random hold-out split per year.